# Time series

Annual global time series from the HAMOCC annual history files (`hbgcy`): volume-weighted mean concentrations (full depth and 0-100 m) and area-integrated fluxes.

In [ ]:
import matplotlib.pyplot as plt
import yaml
from pyprojroot import here

from hamocc import compare, io, plots, timeseries, variables

In [ ]:
# Optional second case for comparison (defaults to the recipe's `experiment_cntl` control case, if any). year_offset aligns its model years with the test case.
caseid2 = None
year_start2 = None
year_end2 = None
year_offset2 = 0

In [ ]:
with open(here('notebooks_hamocc/settings.yml')) as f:
    cfg = yaml.safe_load(f)
exp = cfg['experiment']
grid_file = cfg['grids'][exp['grid']]['grid_file']

cntl = compare.control_case(cfg)
if caseid2 is None and cntl is not None:
    caseid2, (year_start2, year_end2) = cntl['caseid'], io.period(cntl, 'ts')
    input_root2 = cntl['input_root']
else:
    input_root2 = exp['input_root']

def load_case(caseid, ys, ye, input_root=exp['input_root']):
    files = io.history_files(input_root, caseid, 'hbgcy', ys, ye)
    return io.open_available(files, cfg['variables']['hbgcy'])[0]

cases = [(exp['caseid'], load_case(exp['caseid'], *io.period(exp, 'ts')), 0)]
if caseid2:
    cases.append((caseid2, load_case(caseid2, year_start2, year_end2, input_root2), year_offset2))

parea = timeseries.load_parea(grid_file)
labels = [c[0] for c in cases]

def plottable(names):
    """The names every case can supply; prints what is skipped."""
    skipped = {}
    for _, ds, _ in cases:
        names, sk = io.usable(ds, names, timeseries.history_names)
        skipped.update(sk)
    io.report_skipped(skipped)
    return names

def grid_axes(n):
    """3x2 axes with the unused ones hidden."""
    fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, axes.flat[:n]

## Global mean concentrations

Volume-weighted over the full ocean (weights: z-level thickness from `depth_bnds` times cell area, using the z-level fields `*lvl`).

In [ ]:
conc_names = plottable(list(variables.CONC_TS_VARS))
fig, axes = grid_axes(len(conc_names))
for ax, name in zip(axes, conc_names):
    series = [timeseries.global_mean_series(ds, name, parea, offset=off).load() for _, ds, off in cases]
    plots.plot_series(series, labels, ax=ax)
fig.tight_layout()

## Upper-ocean (0-100 m) mean concentrations

From the z-level fields (`*lvl`), layers clipped to 0-100 m.

In [ ]:
fig, axes = grid_axes(len(conc_names))
for ax, name in zip(axes, conc_names):
    series = [timeseries.global_mean_series(ds, name, parea, zmax=100.0, offset=off).load() for _, ds, off in cases]
    plots.plot_series(series, labels, ax=ax, title=f"{series[0].attrs['long_name']}, 0-100 m")
fig.tight_layout()

## Area-integrated fluxes

Annual totals over the global ocean. The net CO2 flux is positive into the ocean (`co2fxd - co2fxu`).

In [ ]:
flux_names = plottable(['co2fxn', 'ppint', 'epc100', 'epcalc100', 'dmsflux'])
if 'ppint' not in flux_names:  # legacy: use pp_tot when ppint is absent
    flux_names = flux_names[:1] + plottable(['pp_tot']) + flux_names[1:]
fig, axes = grid_axes(len(flux_names))
for ax, name in zip(axes, flux_names):
    series = [timeseries.flux_series(ds, name, parea, offset=off).load() for _, ds, off in cases]
    plots.plot_series(series, labels, ax=ax)
fig.tight_layout()